# AO3 Tag Fandom Labeling

Labels rows of an existing tag CSV (e.g. `ao3_tag_clusters.csv`, from
`ao3_tag_analysis.py`/`ao3_tag_analysis.ipynb`) with the fandom(s) each tag is
associated with, computed from real co-occurrence in `ao3_tag_metadata.csv`.

For each `tag_id`, finds every work that contains it and looks at what
fandom(s) those works belong to, reporting the top N fandoms by co-occurrence
percentage (e.g. "Fandom A (62%), Fandom B (25%), Fandom C (13%)"). This is
descriptive co-occurrence computed directly from the scrape, not a guess from
the tag's name -- a fandom-field tag will trivially show itself near 100%
(works tagged with a fandom are, almost by definition, in that fandom); a
cross-cutting trope tag like "Angst" will show a spread across many fandoms
instead.

No network access is required -- this only reads local CSVs.

In [ ]:
# Installs any of this notebook's dependencies that aren't already present.
# Safe to re-run.
%pip install -q pandas

In [ ]:
import pandas as pd

## Configuration

In [ ]:
INPUT = "ao3_tag_metadata.csv"
CLUSTERS_CSV = "ao3_tag_clusters.csv"   # tag CSV to label -- must have a tag_id column
TOP_N = 3                               # number of top co-occurring fandoms to report per tag
COLUMN_NAME = "top_fandoms"             # name for the new fandom-label column
OUT = "ao3_tag_clusters_with_fandoms.csv"   # written as a new file -- CLUSTERS_CSV is
                                             # never overwritten
CLUSTER_FANDOMS_OUT = "ao3_cluster_fandoms.csv"   # per-cluster fandom summary -- skipped
                                                   # if CLUSTERS_CSV has no cluster_id column

DELIMITER = ", "
ALL_METADATA_FIELDS = ["rating", "warnings", "category", "fandom",
                        "relationship", "character", "additional_tags"]

## Load metadata

In [ ]:
# copied from ao3_tag_visualizer.py -- keep in sync if that file changes
def load_metadata(input_csv):
    df = pd.read_csv(input_csv, dtype=str, keep_default_na=False)
    return df


def split_values(cell, delimiter=DELIMITER):
    if not cell:
        return []
    return [v.strip() for v in cell.split(delimiter) if v.strip()]


def explode_field(df, field):
    exploded = df[["work_id", field]].copy()
    exploded[field] = exploded[field].apply(split_values)
    exploded = exploded.explode(field)
    exploded = exploded[exploded[field].notna() & (exploded[field] != "")]
    return exploded


df = load_metadata(INPUT)
df.head()

## Fandom labeling

`compute_fandom_labels` builds a `(work_id, tag_id)` table across all seven
metadata fields, joins it against each work's `fandom` value(s), and reports
each tag's top-N co-occurring fandoms by percentage. The percentage
denominator is the tag's total document count (every work containing the
tag), not just the subset with a known fandom, so a tag whose usage is partly
on fandom-less works will show percentages that don't sum to 100 -- an honest
reflection of the data rather than silently renormalizing it away. Crossover
works with multiple fandoms can likewise push a tag's percentages to sum
above 100, for the same reason (fandom is a multi-valued field, same as every
other field this codebase pools).

Ranking/truncation uses vectorized pandas ops (`sort_values` +
`groupby().head()`) over the whole frame at once, not a Python-level loop per
tag -- important at `--all-tags` scale (tens of thousands of tags), where a
per-tag loop was confirmed to take 32s at ~26,000 tags versus under a second
here.

In [ ]:
# copied from ao3_tag_analysis.py -- keep in sync if that file changes
def build_document_tag_table(df, fields=ALL_METADATA_FIELDS):
    deduped = df.drop_duplicates(subset="work_id", keep="first")
    tables = []
    for field in fields:
        exploded = explode_field(deduped, field)
        exploded = exploded.rename(columns={field: "value"})
        exploded["tag_id"] = field + "::" + exploded["value"]
        tables.append(exploded[["work_id", "tag_id"]])
    return pd.concat(tables, ignore_index=True)


# copied from ao3_tag_fandom_labels.py -- keep in sync if that file changes
def compute_fandom_summary(df, tag_ids, top_n):
    """Returns a DataFrame [tag_id, n_fandoms, top_fandoms], one row per
    tag_id, computed from real per-work co-occurrence in df:

      - top_fandoms: "Fandom A (62%), Fandom B (25%), ..." -- the top_n
        co-occurring fandoms, ranked by descending co-occurrence percentage
        (tie-break: alphabetically smallest fandom name). The percentage
        denominator is the tag's total document count (every work
        containing the tag), not just the subset with a known fandom, so a
        tag whose usage is partly on fandom-less works will show
        percentages that don't sum to 100 -- an honest reflection of the
        data rather than silently renormalizing it away. Crossover works
        with multiple fandoms can likewise push a tag's percentages to sum
        above 100, for the same reason (fandom is a multi-valued field,
        same as every other field this codebase pools).
      - n_fandoms: how many DISTINCT fandoms co-occur with the tag at all
        (every fandom that appears at least once, NOT truncated to top_n)
        -- so you can see both the top few and how concentrated vs. spread
        the tag's fandom usage is (a character tag might span 2 fandoms, a
        cross-cutting trope like Angst 200).

    tag_ids absent from df's metadata entirely (a stale/mismatched
    --clusters-csv) get n_fandoms=0 and an empty top_fandoms rather than
    raising."""
    tag_table = build_document_tag_table(df, fields=ALL_METADATA_FIELDS)
    tag_table = tag_table[tag_table["tag_id"].isin(tag_ids)]
    tag_totals = tag_table.groupby("tag_id").size()

    # Dedupe by work_id before exploding, exactly as build_document_tag_table
    # does internally for the numerator's other side -- the scraper emits one
    # row per (seed tag, work), so a work found via 3 seed tags appears 3
    # times in df. Without this dedup those duplicate rows each contribute
    # the work's fandom to the merge below while tag_totals (built from the
    # deduped table) counts the work once, inflating percentages up to
    # 300%+ on real data.
    deduped = df.drop_duplicates(subset="work_id", keep="first")
    fandom_table = explode_field(deduped, "fandom")[["work_id", "fandom"]]

    merged = tag_table.merge(fandom_table, on="work_id")
    counts = merged.groupby(["tag_id", "fandom"]).size().reset_index(name="count")
    counts["pct"] = counts["count"] / counts["tag_id"].map(tag_totals) * 100

    n_fandoms = counts.groupby("tag_id")["fandom"].nunique()

    # Rank/truncate with vectorized pandas ops (sort_values + groupby().head()),
    # not a Python-level loop over every tag_id's group -- at real --all-tags
    # scale (tens of thousands of tags) a per-group loop calling
    # assign/sort_values/head/itertuples paid pandas' per-call overhead once
    # per tag_id and took 32s at 26,427 tags; this version does the same
    # ranking in 0.9s, confirmed to produce identical output.
    counts = counts.sort_values(["tag_id", "count", "fandom"], ascending=[True, False, True])
    top = counts.groupby("tag_id", sort=False).head(top_n)
    top = top.assign(entry=top["fandom"] + " (" + top["pct"].round(0).astype(int).astype(str) + "%)")
    labels = top.groupby("tag_id", sort=False)["entry"].apply(", ".join)

    ordered = list(tag_ids)
    return pd.DataFrame({
        "tag_id": ordered,
        "n_fandoms": [int(n_fandoms.get(tag_id, 0)) for tag_id in ordered],
        "top_fandoms": [labels.get(tag_id, "") for tag_id in ordered],
    })


def compute_fandom_labels(df, tag_ids, top_n):
    """Thin wrapper over compute_fandom_summary preserving the original
    dict[tag_id -> "Fandom A (62%), ..."] contract (empty string for a
    tag_id absent from df). See compute_fandom_summary for the co-occurrence
    semantics."""
    summary = compute_fandom_summary(df, tag_ids, top_n)
    return dict(zip(summary["tag_id"], summary["top_fandoms"]))


clusters_df = pd.read_csv(CLUSTERS_CSV, dtype=str, keep_default_na=False)
tag_ids = set(clusters_df["tag_id"])
summary = compute_fandom_summary(df, tag_ids, TOP_N)
labels = dict(zip(summary["tag_id"], summary["top_fandoms"]))
fandom_counts = dict(zip(summary["tag_id"], summary["n_fandoms"]))

missing = sum(1 for tag_id in tag_ids if labels.get(tag_id) == "")
if missing:
    print(f"  warning: {missing} of {len(tag_ids)} tags in {CLUSTERS_CSV} had no "
          f"fandom co-occurrence found in {INPUT} (stale/mismatched input?)")

clusters_df["n_fandoms"] = clusters_df["tag_id"].map(fandom_counts)
clusters_df[COLUMN_NAME] = clusters_df["tag_id"].map(labels)
clusters_df.to_csv(OUT, index=False)
print(f"wrote {OUT} ({len(clusters_df)} rows labeled)")
clusters_df.head(20)

## Per-cluster fandom summary

The per-tag labels above answer "which fandoms does *this tag* appear in";
this summarizes at the cluster level instead: for each `cluster_id`, pools
every work containing *any* of the cluster's tags (counted once per cluster,
no matter how many of its tags a work matches) and ranks the fandoms of those
works by percent of works. Same denominator semantics as the per-tag labels:
fandom-less works keep sums under 100, multi-fandom crossover works can push
sums above 100, and every individual value stays <= 100.

In [ ]:
# copied from ao3_tag_analysis.py (the function's home module -- 
# ao3_tag_fandom_labels.py re-exports it) -- keep in sync if it changes
def compute_cluster_fandom_summary(df, clusters_df, top_n):
    """Per-cluster fandom summary (lives here rather than in
    ao3_tag_fandom_labels.py -- which re-exports it -- because the cluster
    meta-network below needs the same labels, and that module imports this
    one, so the import can't go the other way): for each cluster_id, pools
    every work containing ANY of the cluster's tags -- counted once per
    cluster no matter how many of its tags the work matches -- and ranks
    the fandoms of those works by percent of works (tie-break:
    alphabetically smallest fandom name). Same denominator semantics as
    ao3_tag_fandom_labels.py's per-tag labels: the percentage base is the
    cluster's full work pool, so fandom-less works keep sums under 100 and
    multi-fandom crossover works can push sums above 100, while every
    individual value stays <= 100. Returns a DataFrame
    [cluster_id, n_tags, n_works, n_fandoms, top_fandoms] with one row per
    cluster in clusters_df -- a cluster whose tags never appear in df keeps
    its row with n_works=0, n_fandoms=0, and an empty label rather than
    vanishing. n_fandoms is how many DISTINCT fandoms the cluster's works
    span (every fandom that appears at least once, not truncated to
    top_n)."""
    cluster_by_tag = clusters_df.drop_duplicates("tag_id").set_index("tag_id")["cluster_id"]

    tag_table = build_document_tag_table(df, fields=ALL_METADATA_FIELDS)
    tag_table = tag_table[tag_table["tag_id"].isin(cluster_by_tag.index)]
    cluster_works = tag_table.assign(cluster_id=tag_table["tag_id"].map(cluster_by_tag))
    # A work with several of the cluster's tags is still one story.
    cluster_works = cluster_works[["cluster_id", "work_id"]].drop_duplicates()
    cluster_totals = cluster_works.groupby("cluster_id").size()

    # Deduped for the same reason as compute_fandom_labels: the scraper
    # emits one row per (seed tag, work).
    deduped = df.drop_duplicates(subset="work_id", keep="first")
    fandom_table = explode_field(deduped, "fandom")[["work_id", "fandom"]]

    merged = cluster_works.merge(fandom_table, on="work_id")
    counts = merged.groupby(["cluster_id", "fandom"]).size().reset_index(name="count")
    counts["pct"] = counts["count"] / counts["cluster_id"].map(cluster_totals) * 100

    cluster_fandoms = counts.groupby("cluster_id")["fandom"].nunique()

    counts = counts.sort_values(["cluster_id", "count", "fandom"], ascending=[True, False, True])
    top = counts.groupby("cluster_id", sort=False).head(top_n)
    top = top.assign(entry=top["fandom"] + " (" + top["pct"].round(0).astype(int).astype(str) + "%)")
    labels = top.groupby("cluster_id", sort=False)["entry"].apply(", ".join)

    n_tags = clusters_df.groupby("cluster_id")["tag_id"].nunique()
    summary = pd.DataFrame({
        "cluster_id": n_tags.index,
        "n_tags": n_tags.values,
        "n_works": n_tags.index.map(cluster_totals).fillna(0).astype(int),
        "n_fandoms": n_tags.index.map(cluster_fandoms).fillna(0).astype(int),
        "top_fandoms": n_tags.index.map(labels).fillna(""),
    })
    # ao3_tag_clusters.csv's cluster_ids are integers, but they arrive as
    # strings when the CSV is read with dtype=str -- order numerically when
    # every id parses as a number, so 2 sorts before 10.
    numeric_order = pd.to_numeric(summary["cluster_id"], errors="coerce")
    if numeric_order.notna().all():
        summary = summary.iloc[numeric_order.argsort(kind="stable").to_numpy()]
    else:
        summary = summary.sort_values("cluster_id")
    return summary.reset_index(drop=True)


if "cluster_id" in clusters_df.columns:
    cluster_summary = compute_cluster_fandom_summary(df, clusters_df, TOP_N)
    cluster_summary.to_csv(CLUSTER_FANDOMS_OUT, index=False)
    print(f"wrote {CLUSTER_FANDOMS_OUT} ({len(cluster_summary)} clusters)")
    display(cluster_summary.head(20))
else:
    print(f"note: {CLUSTERS_CSV} has no cluster_id column -- skipping "
          f"the per-cluster fandom summary")

## Done

`{OUT}` (per-tag labels) and `{CLUSTER_FANDOMS_OUT}` (per-cluster summary) are
now in the notebook's working directory. Re-run from the Configuration cell
with a different `TOP_N`/`COLUMN_NAME`/`CLUSTERS_CSV`/`OUT`/
`CLUSTER_FANDOMS_OUT` to relabel.